In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
file_path = (r"C:\Users\ACER\Documents\ExcelR Projects\Insurance Project\Insurance_Analytics_Project\data\raw\Policy Data.xlsx")

customer_df = pd.read_excel(
    file_path,
    sheet_name="Customer Information"
)

policy_df = pd.read_excel(
    file_path,
    sheet_name="Policy Details"
)

claims_df = pd.read_excel(
    file_path,
    sheet_name="Claims"
)

payment_df = pd.read_excel(
    file_path,
    sheet_name="Payment History"
)

additional_df = pd.read_excel(
    file_path,
    sheet_name="Additional Fields"
)

In [3]:
# Data Type Standardization

customer_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Customer ID                      5000 non-null   object
 1   Name                             5000 non-null   object
 2   Gender                           5000 non-null   object
 3   Age                              5000 non-null   int64 
 4   Occupation                       5000 non-null   object
 5   Marital Status                   5000 non-null   object
 6   Address (City, State, Zip Code)  5000 non-null   object
dtypes: int64(1), object(6)
memory usage: 273.6+ KB


In [4]:
policy_df["Policy Start Date"] = pd.to_datetime(
    policy_df["Policy Start Date"]
)

policy_df["Policy End Date"] = pd.to_datetime(
    policy_df["Policy End Date"]
)

In [5]:
claims_df["Date of Claim"] = pd.to_datetime(
    claims_df["Date of Claim"]
)

claims_df["Settlement Date"] = pd.to_datetime(
    claims_df["Settlement Date"]
)

In [6]:
payment_df["Date of Payment"] = pd.to_datetime(
    payment_df["Date of Payment"]
)

In [7]:
policy_df.info()

claims_df.info()

payment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Policy ID          5000 non-null   object        
 1   Policy Type        5000 non-null   object        
 2   Coverage Amount    5000 non-null   int64         
 3   Premium Amount     5000 non-null   float64       
 4   Policy Start Date  5000 non-null   datetime64[ns]
 5   Policy End Date    5000 non-null   datetime64[ns]
 6   Payment Frequency  5000 non-null   object        
 7   Status             5000 non-null   object        
 8   Customer ID        5000 non-null   object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(5)
memory usage: 351.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----  

In [8]:
# Business Rule Validation

invalid_age = customer_df[
    (customer_df["Age"] < 18)
    |
    (customer_df["Age"] > 100)
]

print("Invalid Ages:", len(invalid_age))

Invalid Ages: 0


In [9]:
invalid_coverage = policy_df[
    policy_df["Coverage Amount"] <= 0
]

print(
    "Invalid Coverage:",
    len(invalid_coverage)
)

Invalid Coverage: 0


In [10]:
invalid_premium = policy_df[
    policy_df["Premium Amount"] <= 0
]

print(
    "Invalid Premium:",
    len(invalid_premium)
)

Invalid Premium: 0


In [11]:
invalid_risk = additional_df[
    (additional_df["Risk Score"] < 0)
    |
    (additional_df["Risk Score"] > 100)
]

print(
    "Invalid Risk Score:",
    len(invalid_risk)
)

Invalid Risk Score: 0


In [12]:
invalid_policy_dates = policy_df[
    policy_df["Policy Start Date"]
    >
    policy_df["Policy End Date"]
]

print(
    "Invalid Policy Dates:",
    len(invalid_policy_dates)
)

Invalid Policy Dates: 0


In [13]:
invalid_claim_dates = claims_df[
    claims_df["Settlement Date"]
    <
    claims_df["Date of Claim"]
]

print(
    "Invalid Claim Dates:",
    len(invalid_claim_dates)
)

Invalid Claim Dates: 0


In [14]:
# Data Quality Flags

claims_df["Settlement_Date_Missing_Flag"] = (
    claims_df["Settlement Date"]
    .isnull()
    .astype(int)
)

In [19]:
claim_threshold = claims_df[
    "Claim Amount"
].median()

claims_df["High_Claim_Flag"] = np.where(
    claims_df["Claim Amount"] >
    claim_threshold,
    1,
    0
)

In [20]:
# Payments Table

payment_df["Failed_Payment_Flag"] = np.where(
    payment_df["Payment Status"] ==
    "Failed",
    1,
    0
)

In [21]:
additional_df["High_Risk_Flag"] = np.where(
    additional_df["Risk Score"] >= 70,
    1,
    0
)

In [22]:
# Standardized Categories

def risk_category(score):

    if score <= 30:
        return "Low Risk"

    elif score <= 60:
        return "Medium Risk"

    else:
        return "High Risk"

In [23]:
additional_df["Risk Category"] = (
    additional_df["Risk Score"]
    .apply(risk_category)
)

In [24]:
def age_bucket(age):

    if age <= 25:
        return "18-25"

    elif age <= 35:
        return "26-35"

    elif age <= 45:
        return "36-45"

    elif age <= 55:
        return "46-55"

    else:
        return "56+"

In [25]:
customer_df["Age Bucket"] = (
    customer_df["Age"]
    .apply(age_bucket)
)

In [26]:
# Derived Metrics

policy_df["Policy Duration Days"] = (
    policy_df["Policy End Date"]
    -
    policy_df["Policy Start Date"]
).dt.days

In [27]:
claims_df["Settlement Days"] = (
    claims_df["Settlement Date"]
    -
    claims_df["Date of Claim"]
).dt.days

In [28]:
payment_df["Payment Year"] = (
    payment_df["Date of Payment"]
).dt.year

In [29]:
payment_df["Payment Month"] = (
    payment_df["Date of Payment"]
).dt.month_name()

In [30]:
# Data Quality Scorecard

dq_scorecard = pd.DataFrame({
    "Validation": [
        "Invalid Age",
        "Invalid Coverage",
        "Invalid Premium",
        "Invalid Risk Score",
        "Invalid Policy Dates",
        "Invalid Claim Dates"
    ],
    "Count": [
        len(invalid_age),
        len(invalid_coverage),
        len(invalid_premium),
        len(invalid_risk),
        len(invalid_policy_dates),
        len(invalid_claim_dates)
    ]
})

dq_scorecard

,Validation,Count
0,Invalid Age,0
1,Invalid Coverage,0
2,Invalid Premium,0
3,Invalid Risk Score,0
4,Invalid Policy Dates,0
5,Invalid Claim Dates,0


In [31]:
# Export Cleaned Datasets

customer_df.to_csv(
    "../data/processed/cleaned_customer.csv",
    index=False
)

policy_df.to_csv(
    "../data/processed/cleaned_policy.csv",
    index=False
)

claims_df.to_csv(
    "../data/processed/cleaned_claims.csv",
    index=False
)

payment_df.to_csv(
    "../data/processed/cleaned_payments.csv",
    index=False
)

additional_df.to_csv(
    "../data/processed/cleaned_additional.csv",
    index=False
)

In [32]:
dq_scorecard.to_excel(
    "../data/reports/data_quality_scorecard.xlsx",
    index=False
)